In [1]:
import pandas as pd

deepseek_v3_df = pd.read_csv(
    "../../data/out/distillation/mmlu_deepseek_v3.tsv",
    sep="\t",
    header=0,
)
qwen3_235b_df = pd.read_csv(
    "../../data/out/distillation/mmlu_qwen3_235b.tsv",
    sep="\t",
    header=0,
)

In [2]:
qwen3_235b_df.describe()

,cot_content,question_id,answer_index,total_tokens,distill_answer
count,0.0,12032.000000,12032.000000,12032.000000,10868.000000
mean,NaN,6168.925033,4.200216,162.544132,5.202705
std,NaN,3519.353467,2.854734,117.248942,2.840285
min,NaN,70.000000,0.000000,23.000000,1.000000
25%,NaN,3127.750000,2.000000,81.000000,3.000000
50%,NaN,6176.500000,4.000000,128.000000,5.000000
75%,NaN,9212.250000,7.000000,206.000000,8.000000
max,NaN,12256.000000,9.000000,1195.000000,10.000000


In [3]:
qwen3_235b_df.value_counts('distill_ans_correct')

distill_ans_correct
True     9527
False    2505
Name: count, dtype: int64

In [4]:
deepseek_v3_df_true_only = deepseek_v3_df[deepseek_v3_df["distill_ans_correct"]]
qwen3_235b_df_true_only = qwen3_235b_df[qwen3_235b_df["distill_ans_correct"]]

In [5]:
indices_to_drop = deepseek_v3_df_true_only.index.intersection(qwen3_235b_df_true_only.index)
deepseek_v3_df_addition = deepseek_v3_df_true_only.drop(indices_to_drop, errors="ignore")

In [6]:
deepseek_v3_df_addition.describe()

,cot_content,question_id,answer_index,total_tokens,distill_answer
count,0.0,846.000000,846.000000,846.000000,846.000000
mean,NaN,7156.099291,4.124113,201.791962,5.124113
std,NaN,3986.050782,2.836359,135.537202,2.836359
min,NaN,109.000000,0.000000,30.000000,1.000000
25%,NaN,3779.750000,2.000000,104.000000,3.000000
50%,NaN,7953.500000,4.000000,159.000000,5.000000
75%,NaN,11282.500000,6.000000,269.000000,7.000000
max,NaN,12254.000000,9.000000,1033.000000,10.000000


In [7]:
# Create a copy to avoid modifying the original
qwen3_235b_df_merged = qwen3_235b_df.copy()

# Set the question_id as index for easier updating
qwen3_235b_df_merged.set_index('question_id', inplace=True)
deepseek_addition_indexed = deepseek_v3_df_addition.set_index('question_id')

# Update the rows where question_ids match
qwen3_235b_df_merged.update(deepseek_addition_indexed[["distill_ans_correct", "distill_response", "distill_answer"]])

# Reset index back to default
qwen3_235b_df_merged.reset_index(inplace=True)

In [8]:
qwen3_235b_df_merged.describe()

,question_id,cot_content,answer_index,total_tokens,distill_answer
count,12032.000000,0.0,12032.000000,12032.000000,11449.000000
mean,6168.925033,NaN,4.200216,162.544132,5.190148
std,3519.353467,NaN,2.854734,117.248942,2.843935
min,70.000000,NaN,0.000000,23.000000,1.000000
25%,3127.750000,NaN,2.000000,81.000000,3.000000
50%,6176.500000,NaN,4.000000,128.000000,5.000000
75%,9212.250000,NaN,7.000000,206.000000,8.000000
max,12256.000000,NaN,9.000000,1195.000000,10.000000


In [9]:
qwen3_235b_df_merged.value_counts('distill_ans_correct')

distill_ans_correct
True     10373
False     1659
Name: count, dtype: int64

In [10]:
qwen3_235b_df_merged.columns

Index(['question_id', 'src', 'answer', 'options', 'category', 'question',
       'cot_content', 'answer_index', 'total_tokens', 'meta_cluster',
       'base_cluster', 'distill_ans_correct', 'distill_response',
       'distill_answer'],
      dtype='object')

In [11]:
qwen3_235b_df_merged.to_csv('../../data/out/distillation/mmlu_merged.tsv', sep="\t", index=False)